# 03 · Store Demand Intelligence: EDA and Preprocessing

**Dataset:** `data/raw/demand/retail_store_inventory.csv` (73,100 rows, one row per store × product × date).

**Output:** `data/cleaned/demand/store_demand_cleaned.csv`, read by `src/models/train_demand.py` and the Store Demand page.

**Target:** `Units Sold`. `Demand Forecast` is a column in the source data, but it is **not** used as a model input, because it may already contain information the model should not learn from.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

# Works whether Jupyter is opened from the project root or from the notebooks/ folder.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("Project root:", ROOT)

## 1. Load and inspect

In [ ]:
RAW_PATH = ROOT / "data" / "raw" / "demand" / "retail_store_inventory.csv"
CLEAN_PATH = ROOT / "data" / "cleaned" / "demand" / "store_demand_cleaned.csv"

df_raw = pd.read_csv(RAW_PATH)
print("Shape:", df_raw.shape)
print("Columns:", list(df_raw.columns))
df_raw.info()
display(df_raw.head())

## 2. Missing values, duplicates and unique values

In [ ]:
print("Missing values:")
print(df_raw.isna().sum())
print("\nExact duplicate rows:", df_raw.duplicated().sum())
for col in ["Store ID", "Product ID", "Category", "Region", "Weather Condition", "Seasonality"]:
    print(f"{col}: {df_raw[col].nunique()} unique -> {sorted(df_raw[col].dropna().unique())[:8]}")

## 3. Date coverage

In [ ]:
dates = pd.to_datetime(df_raw["Date"], errors="coerce")
print("Unparsable dates:", dates.isna().sum())
print("Date range:", dates.min().date(), "to", dates.max().date())
print("Days covered:", dates.nunique())

## 4. Numerical validation

Negative sales or stock would be impossible, so they are reported and removed. Zero sales are valid (a product simply did not sell that day) and are kept.

In [ ]:
num_cols = ["Inventory Level", "Units Sold", "Units Ordered", "Demand Forecast",
            "Price", "Discount", "Holiday/Promotion", "Competitor Pricing"]
display(df_raw[num_cols].describe().T.round(2))
print("Negative Units Sold:", (df_raw["Units Sold"] < 0).sum())
print("Negative Inventory Level:", (df_raw["Inventory Level"] < 0).sum())
print("Zero Units Sold:", (df_raw["Units Sold"] == 0).sum())

## 5. Cleaning decisions

1. **Parse `Date`** and keep rows where it is valid. The model later builds year, month and weekday from it.
2. **Strip text** columns.
3. **Convert numeric columns**, with unreadable values becoming missing.
4. **Remove rows with a missing target** (`Units Sold`) or missing date. These cannot be used to train a supervised model.
5. **Remove exact duplicates.**
6. **Save `Date` as `YYYY-MM-DD` text**, the format the training code expects.

In [ ]:
df = df_raw.copy()
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

for col in ["Store ID", "Product ID", "Category", "Region", "Weather Condition", "Seasonality"]:
    df[col] = df[col].astype("string").str.strip()
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

before = len(df)
df = df.dropna(subset=["Date", "Units Sold"])
df = df[df["Units Sold"] >= 0]
df = df.drop_duplicates()
print("Rows removed:", before - len(df))

df["Date"] = df["Date"].dt.strftime("%Y-%m-%d")
print("Rows after cleaning:", len(df))

## 6. Save the cleaned dataset

In [ ]:
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_PATH, index=False)
print("Saved cleaned dataset:", CLEAN_PATH)

## 7. EDA: units sold by product category and by weather

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
df.groupby("Category")["Units Sold"].mean().sort_values().plot(kind="barh", ax=axes[0], color="#3B8F7B")
axes[0].set_title("Average units sold by category")
df.groupby("Weather Condition")["Units Sold"].mean().sort_values().plot(kind="barh", ax=axes[1], color="#A7F432")
axes[1].set_title("Average units sold by weather")
plt.tight_layout()
plt.show()